# 03 — Classical and volatility-aware innovation benchmarks

Fit five interpretable residual laws on rolling calibration errors. We retain an IID bootstrap for comparison, add a circular block bootstrap to preserve short-run residual dependence, and add an EWMA-standardized bootstrap for changing volatility. The untouched test period is evaluated at repeated forecast origins.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)

from innovcal.experiments.financial import run_financial_experiment
from innovcal.forecasting.rolling import generate_rolling_var_forecasts

In [2]:
returns = pd.read_csv(ROOT / 'data/processed/financial_returns.csv', index_col=0).to_numpy()
METHODS = ('gaussian', 'student_t', 'bootstrap', 'block_bootstrap', 'volatility_bootstrap')
result = run_financial_experiment(
    returns,
    methods=METHODS,
    lags=1,
    n_paths=250,
    seed=123,
    block_length=10,
    volatility_span=60,
)
display(result.evaluation.sort_values('energy_score'))

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: divide by zero encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: overflow encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: invalid value encountered in matmul
  shocks = rng.

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,...,width_1,coverage_2,width_2,coverage_3,width_3,coverage_4,width_4,nominal_coverage,coverage_error,abs_coverage_error
3,financial,VAR,block_bootstrap,0.919801,0.055404,0.019848,0.008351,0.074207,0.025636,0.009509,...,0.062101,0.935214,0.057658,0.947753,0.065146,0.887147,0.036712,0.9,0.019801,0.019801
2,financial,VAR,bootstrap,0.925810,0.055850,0.019887,0.008376,0.075100,0.031034,0.011536,...,0.062471,0.941484,0.058372,0.944619,0.065166,0.900731,0.037391,0.9,0.025810,0.025810
1,financial,VAR,student_t,0.937565,0.059787,0.020001,0.008436,0.075828,0.074138,0.022633,...,0.064769,0.957158,0.064964,0.948798,0.067052,0.925810,0.042364,0.9,0.037565,0.037565
4,financial,VAR,volatility_bootstrap,0.940961,0.061266,0.020133,0.008484,0.076905,0.074312,0.022163,...,0.062924,0.966562,0.070019,0.958203,0.070539,0.923720,0.041582,0.9,0.040961,0.040961
0,financial,VAR,gaussian,0.949321,0.063124,0.020433,0.008642,0.077406,0.111233,0.036207,...,0.068334,0.966562,0.068714,0.957158,0.070868,0.937304,0.044582,0.9,0.049321,0.049321


In [3]:
test_start = len(result.split.train) + len(result.split.calibration)
rolling = generate_rolling_var_forecasts(
    returns,
    test_start=test_start,
    innovation_models=result.innovation_models,
    horizon=20,
    n_paths=250,
    lags=1,
    origin_step=20,
    seed=123,
)
print(f'{len(rolling.origins)} complete rolling forecast origins')

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: divide by zero encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: overflow encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: invalid value encountered in matmul
  shocks = rng.

47 complete rolling forecast origins


In [4]:
result.evaluation.to_csv(CACHE / 'classical_evaluation.csv', index=False)
np.savez(
    CACHE / 'financial_split.npz',
    train=result.split.train,
    calibration=result.split.calibration,
    test=result.split.test,
)
np.savez(
    CACHE / 'fitted_var.npz',
    beta=result.fitted_var['beta'],
    lags=result.fitted_var['lags'],
    residuals=result.residuals,
)
for model, forecast in result.forecasts.items():
    np.savez(CACHE / f'forecast_{model}.npz', **forecast)
for model, tensor in rolling.forecasts.items():
    np.savez(
        CACHE / f'rolling_{model}.npz',
        forecasts=tensor,
        observations=rolling.observations,
        origins=rolling.origins,
    )